In [2]:
import os
import yaml
import sys
import keras
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_path)
from types import SimpleNamespace
from src.config_preprocessing import normalize_config
from src.train_preprocessing import run_train

# Load configuration from YAML file
config_path = os.path.join(root_path, "configs/train_config.yaml")
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config file not found: {config_path}")
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg = normalize_config(cfg)
args = SimpleNamespace(**cfg)

In [ ]:
print("Attempting to load model without custom_objects...")
model = keras.saving.load_model(weights_path, compile=False)

print("Model loaded:", getattr(model, 'name', str(model)))

# Ensure variables are created (subclassed models may be unbuilt)
try:
    weight_count = len(model.weights)
except Exception:
    weight_count = 0

if weight_count == 0:
    print("Model has no weights yet — attempting to build or run a dummy forward pass to force variable creation...")
    h = getattr(args, 'height', 480)
    w = getattr(args, 'width', 640)
    c = 3
    batch = 1
    try:
        input_shape = {
            "left_input": (batch, h, w, c),
            "right_input": (batch, h, w, c),
            "disp_map": (batch, h, w, 1),
        }
        print("Calling model.build with input_shape=", input_shape)
        model.build(input_shape)
    except Exception:
        print("model.build failed; trying a dummy forward pass...")
        dummy_inputs = {
            "left_input": tf.zeros((batch, h, w, c), dtype=tf.float32),
            "right_input": tf.zeros((batch, h, w, c), dtype=tf.float32),
        }
        _ = model(dummy_inputs, training=False)

print("\n\nFinal weight count:", len(model.weights))
for i, w in enumerate(model.weights, 1):
    # name = getattr(w, 'name', f'weight_{i}')
    name = w.path
    try:
        arr = w.numpy()
        shape = arr.shape
        dtype = arr.dtype
        preview = arr.ravel()[:10]
    except Exception:
        shape = getattr(w, 'shape', None)
        dtype = None
        preview = None
    print(f"{i}: {name} shape={shape} dtype={dtype} preview={preview}")



Attempting to load model without custom_objects...
Model loaded: mad_net
Model has no weights yet — attempting to build or run a dummy forward pass to force variable creation...
Calling model.build with input_shape= {'left_input': (1, 480, 640, 3), 'right_input': (1, 480, 640, 3), 'disp_map': (1, 480, 640, 1)}
model.build failed; trying a dummy forward pass...


Final weight count: 196
1: mad_net/feature_norm1/gamma shape=(3,) dtype=float32 preview=[1. 1. 1.]
2: mad_net/feature_norm1/beta shape=(3,) dtype=float32 preview=[0. 0. 0.]
3: mad_net/feature_norm2/gamma shape=(16,) dtype=float32 preview=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
4: mad_net/feature_norm2/beta shape=(16,) dtype=float32 preview=[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
5: mad_net/feature_norm3/gamma shape=(16,) dtype=float32 preview=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
6: mad_net/feature_norm3/beta shape=(16,) dtype=float32 preview=[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
7: mad_net/feature_norm4/gamma shape=(32,) dtype=float32 preview=[1. 1. 1. 1

In [3]:
import os
import yaml
import sys
import keras
import tensorflow as tf
import keras
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_path)
from src.madnet import MADNet
from src.preprocessing import StereoDatasetCreator
from src.losses_and_metrics import Bad3, EndPointError, ReconstructionLoss, SSIMLoss
from src.callbacks import TensorboardImagesCallback
from keras.src.utils import file_utils


model = MADNet(name="mad_net")
left = tf.zeros((args.batch_size, args.height, args.width, 3), dtype=tf.float32)
right = tf.zeros_like(left)
print("Running dummy forward pass to create variables...")
_ = model({"left_input": left, "right_input": right}, training=False)
print("Variables created. Attempting to load weights by_name=True...")
model.load_weights("/home/ubuntu24/repos/madnet-deep-stereo-with-keras/notebooks/model_weights/flying_things/synthetic.h5", by_name=True)

# tf1_ckpt = "/home/ubuntu24/repos/madnet-deep-stereo-with-keras/notebooks/model_weights/flying_things/synthetic.h5"
# tf1_ckpt = keras.models.load_model(tf1_ckpt)

I0000 00:00:1762100716.429178  436588 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13551 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


Running dummy forward pass to create variables...


2025-11-02 18:25:17.542223: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300


Variables created. Attempting to load weights by_name=True...


In [4]:
model.summary()

Model: "mad_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ warp_image_block                │ ?                      │   0 (unbuilt) │
│ (WarpImageBlock)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm1                   │ (10, 480, 640, 3)      │             6 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm2                   │ (10, 240, 320, 16)     │            32 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm3                   │ (10, 240, 320, 16)     │            32 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm4                   │ (10, 120, 160, 32)     │            64 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm5                   │ (10, 120, 160, 32)     │            64 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm6                   │ (10, 60, 80, 64)       │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm7                   │ (10, 60, 80, 64)       │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm8                   │ (10, 30, 40, 96)       │           192 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm9                   │ (10, 30, 40, 96)       │           192 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm10                  │ (10, 15, 20, 128)      │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm11                  │ (10, 15, 20, 128)      │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_norm12                  │ (10, 8, 10, 192)       │           384 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_relu (Activation)         │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (10, 240, 320, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (10, 240, 320, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (10, 120, 160, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv2D)                  │ (10, 120, 160, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 3,834,584 (14.63 MB)

 Trainable params: 3,834,584 (14.63 MB)

 Non-trainable params: 0 (0.00 B)